Runtime → Change runtime type → Hardware accelerator → T4 GPU

In [4]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest-asyncio

^C


Verify that the GPU is available:

In [5]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

CUDA available: False


**Expected output will be similar to:**

CUDA available: True

GPU: Tesla T4

VRAM: 14.56 GB

The exact GPU and VRAM may differ depending on the Colab runtime assigned.

This downloads and loads the Qwen model using 4-bit quantization. This will take some time.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Create the SLM API

In [ ]:
def generate_text(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=400,
        )

    generated_tokens = output[0][inputs["input_ids"].shape[1] :]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

FastAPI endpoint that will be wired to ngrok

In [ ]:
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

app = FastAPI()


class GenerateRequest(BaseModel):
    prompt: str


@app.post("/generate")
def generate(req: GenerateRequest):
    return {"output": generate_text(req.prompt)}

Create/log into an ngrok account and obtain an authtoken.

In [ ]:
from getpass import getpass

NGROK_TOKEN = getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)

Enter your ngrok authtoken: ··········


The token is entered interactively and is not stored in the notebook code.

Start the SLM server and ngrok tunnel

In [ ]:
import threading
import time

from pyngrok import ngrok


# Start FastAPI
def run_server():
    config = uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info",
    )
    server = uvicorn.Server(config)
    server.run()


server_thread = threading.Thread(
    target=run_server,
    daemon=True,
)
server_thread.start()

# Give FastAPI a moment to start
time.sleep(2)

# Create ngrok tunnel
public_url = ngrok.connect(8000)

print("SLM endpoint:", public_url)
print("Swagger:", f"{public_url}/docs")

INFO:     Started server process [2147]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


SLM endpoint: NgrokTunnel: "https://3b20-34-16-160-84.ngrok-free.app" -> "http://localhost:8000"
Swagger: NgrokTunnel: "https://3b20-34-16-160-84.ngrok-free.app" -> "http://localhost:8000"/docs


You will get the SLM endpoint.

Set the ngrok URL as an environment variable before starting/restarting Docker

```
$env:SLM_ENDPOINT_URL="https://xxxxxxxx.ngrok-free.app"
docker compose up -d
```



Testing within Colab

In [ ]:
generate_text("Hello")

'Hello! Nice to meet you. How can I assist you today?'

In [ ]:
import requests

response = requests.post(
    public_url.public_url + "/generate",
    json={"prompt": "Hello"},
)

print(response.status_code)
print(response.text)

INFO:     34.16.160.84:0 - "POST /generate HTTP/1.1" 200 OK
200
{"output":"Hello! Nice to meet you. How can I assist you today?"}
